In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_04_molecular_dynamics.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Lesson 04: Molecular Dynamics: Symplectic Integrators, Shadow Hamiltonians, Cutoffs, and Time Reversibility

_Session efc66fd1 · Estimated time: ~120 minutes · 42 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Formulate Hamilton's equations of motion and understand Liouville's theorem of phase-space volume conservation.
- Derive the basic Verlet, Leapfrog, and Velocity Verlet integration algorithms from forward and backward Taylor expansions.
- Explain why symplectic integrators exhibit bounded, oscillatory energy fluctuations without secular drift via shadow (modified) Hamiltonians.
- Contrast symplectic integrators with high-order non-symplectic integrators (Euler, Runge-Kutta 4th order, Gear predictor-corrector).
- Apply physical timestep selection criteria based on fastest vibrational frequencies ($T_{\text{vib}}/20$) and identify the Verlet resonance instability at $\Delta t = T_{\text{vib}}/\pi$.
- Demonstrate the algebraic time-reversibility of Velocity Verlet and assess the impact of finite-precision floating-point Lyapunov divergence.
- Analyze truncated interatomic potentials, compute the impulsive force delta-spike at $r_c$, and implement potential shifting and force smoothing.


In [2]:
ws.render_section('sec_md_architecture_and_equations')

### The Molecular Dynamics Paradigm

Molecular Dynamics (MD) simulates the deterministic time evolution of a classical $N$-body system by numerically integrating Newton's (or Hamilton's) equations of motion. Given particle positions $\mathbf{r}^N = \{\mathbf{r}_1, \dots, \mathbf{r}_N\}$ and momenta $\mathbf{p}^N = \{\mathbf{p}_1, \dots, \mathbf{p}_N\}$ with interatomic potential $V(\mathbf{r}^N)$, the classical Hamiltonian is:

$$\mathcal{H}(\mathbf{r}^N, \mathbf{p}^N) = \sum_{i=1}^N \frac{|\mathbf{p}_i|^2}{2m_i} + V(\mathbf{r}_1, \dots, \mathbf{r}_N)$$

The equations of motion are given by Hamilton's canonical equations:

$$\dot{\mathbf{r}}_i = \frac{\partial \mathcal{H}}{\partial \mathbf{p}_i} = \frac{\mathbf{p}_i}{m_i} = \mathbf{v}_i, \qquad \dot{\mathbf{p}}_i = -\frac{\partial \mathcal{H}}{\partial \mathbf{r}_i} = -\nabla_i V = \mathbf{F}_i$$

### Liouville's Theorem and Symplectic Geometry

The phase space $\Gamma = (\mathbf{r}^N, \mathbf{p}^N)$ has dimension $6N$. Under exact Hamiltonian dynamics, the phase-space velocity field $\mathbf{u} = (\dot{\mathbf{r}}^N, \dot{\mathbf{p}}^N)$ has zero divergence:

$$\nabla_\Gamma \cdot \mathbf{u} = \sum_{i=1}^N \left( \frac{\partial \dot{\mathbf{r}}_i}{\partial \mathbf{r}_i} + \frac{\partial \dot{\mathbf{p}}_i}{\partial \mathbf{p}_i} \right) = \sum_{i=1}^N \left( \frac{\partial^2 \mathcal{H}}{\partial \mathbf{r}_i \partial \mathbf{p}_i} - \frac{\partial^2 \mathcal{H}}{\partial \mathbf{p}_i \partial \mathbf{r}_i} \right) = 0$$

By the divergence theorem, this implies **Liouville's Theorem**: phase-space volume elements are strictly conserved along trajectories ($d\Omega / dt = 0$). More fundamentally, Hamiltonian flows preserve the differential 2-form $\omega = \sum_i d\mathbf{r}_i \wedge d\mathbf{p}_i$, a geometric property called **symplecticity**. A numerical integrator that preserves this geometric structure is called a **symplectic integrator**.


In [3]:
ws.render_section('sec_integrators_and_symplectic_geometry')

### Derivation of the Verlet Algorithm

Consider the Taylor expansions of position $\mathbf{r}(t)$ forward and backward by timestep $\Delta t$:

$$\mathbf{r}(t + \Delta t) = \mathbf{r}(t) + \mathbf{v}(t)\Delta t + \frac{1}{2}\mathbf{a}(t)\Delta t^2 + \frac{1}{6}\mathbf{b}(t)\Delta t^3 + \mathcal{O}(\Delta t^4)$$
$$\mathbf{r}(t - \Delta t) = \mathbf{r}(t) - \mathbf{v}(t)\Delta t + \frac{1}{2}\mathbf{a}(t)\Delta t^2 - \frac{1}{6}\mathbf{b}(t)\Delta t^3 + \mathcal{O}(\Delta t^4)$$

Adding these two expansions cancels the odd-order terms (velocity and jerk $\mathbf{b}(t)$):

$$\mathbf{r}(t + \Delta t) = 2\mathbf{r}(t) - \mathbf{r}(t - \Delta t) + \mathbf{a}(t)\Delta t^2 + \mathcal{O}(\Delta t^4)$$

The local position truncation error is $\mathcal{O}(\Delta t^4)$, yielding an accumulated global position error of $\mathcal{O}(\Delta t^2)$ over time $T$.

### Velocities and the Velocity Verlet Formulation

Subtracting the Taylor expansions yields a central-difference estimate for velocity with error $\mathcal{O}(\Delta t^2)$:

$$\mathbf{v}(t) = \frac{\mathbf{r}(t + \Delta t) - \mathbf{r}(t - \Delta t)}{2\Delta t} + \mathcal{O}(\Delta t^2)$$

However, the basic Verlet algorithm does not store $\mathbf{v}(t)$ explicitly, making kinetic energy evaluation lag by one step. Swope et al. (1982) introduced the mathematically equivalent **Velocity Verlet** scheme, which maintains positions and velocities synchronized at time $t$:

1. Propagate velocity by half-step: $\mathbf{v}(t + \frac{1}{2}\Delta t) = \mathbf{v}(t) + \frac{1}{2}\mathbf{a}(t)\Delta t$
2. Propagate position by full-step: $\mathbf{r}(t + \Delta t) = \mathbf{r}(t) + \mathbf{v}(t + \frac{1}{2}\Delta t)\Delta t = \mathbf{r}(t) + \mathbf{v}(t)\Delta t + \frac{1}{2}\mathbf{a}(t)\Delta t^2$
3. Compute new forces: $\mathbf{a}(t + \Delta t) = \frac{1}{m}\mathbf{F}(\mathbf{r}(t + \Delta t))$
4. Complete velocity step: $\mathbf{v}(t + \Delta t) = \mathbf{v}(t + \frac{1}{2}\Delta t) + \frac{1}{2}\mathbf{a}(t + \Delta t)\Delta t$

Notice that each step requires exactly **one force evaluation** per particle.


In [4]:
ws.render_section('concept_integrator_shootout')

### Interactive Concept: Integrator Shootout (Verlet vs Euler vs RK4)

Compare numerical integrators on a 1D harmonic oscillator $\ddot{x} + \omega^2 x = 0$.
Select the integrator (`velocity_verlet`, `verlet`, `euler`, `rk4`), adjust timestep $\Delta t$, and track phase space orbits and energy conservation.

**Key Insight:** Forward Euler spirals outward exponentially (violating phase-space volume conservation). RK4 produces extremely small short-term errors but accumulates steady secular energy drift over long simulations. In contrast, Velocity Verlet and Verlet bound energy fluctuations within a narrow band forever because they exactly conserve a shadow Hamiltonian.

In [5]:
ws.render_section('q04_01')

**Q1 · easy** · 💡

What does Liouville's theorem state regarding Hamiltonian phase space flow?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the divergence of the canonical phase-space velocity field $\nabla_\Gamma \cdot \mathbf{u} = 0$.

</div>
</details>

- [ ] **(A)** Total kinetic energy and potential energy are each independently conserved at every point in time.
- [ ] **(B)** The volume of an ensemble of trajectories in $6N$-dimensional phase space is conserved over time ($d\Omega/dt = 0$).
- [ ] **(C)** The distance between any two nearby trajectories in phase space remains strictly constant.
- [ ] **(D)** All microstates in phase space have zero momentum.

In [6]:
ws.render_section('q04_02')

**Q2 · easy** · 💡

In the derivation of the standard Verlet algorithm $\mathbf{r}(t+\Delta t) = 2\mathbf{r}(t) - \mathbf{r}(t-\Delta t) + \mathbf{a}(t)\Delta t^2$, why do the odd-power terms in $\Delta t$ vanish?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider $(+\Delta t)^n + (-\Delta t)^n$ for $n=1, 3, 5$.

</div>
</details>

- [ ] **(A)** The odd derivatives of potential energy are identically zero for all physical potentials.
- [ ] **(B)** Adding forward $\mathbf{r}(t+\Delta t)$ and backward $\mathbf{r}(t-\Delta t)$ Taylor expansions cancels all odd powers of $\Delta t$ by symmetry.
- [ ] **(C)** Newton's second law sets odd time derivatives of position to zero.
- [ ] **(D)** The kinetic energy cancels out the odd terms.

In [7]:
ws.render_section('q04_03')

**Q3 · medium** · 💡

What are the local position truncation error and global position truncation error of the Verlet algorithm over a fixed simulation time $T$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

The lowest discarded term in the summed Taylor series is $\frac{2}{4!} \mathbf{r}^{(4)}(t)\Delta t^4$. Over $N_{\text{steps}} = T/\Delta t$ steps, how does error accumulate?

</div>
</details>

- [ ] **(A)** Local error is $\mathcal{O}(\Delta t^2)$, global error is $\mathcal{O}(\Delta t)$.
- [ ] **(B)** Local error is $\mathcal{O}(\Delta t^3)$, global error is $\mathcal{O}(\Delta t^2)$.
- [ ] **(C)** Local error is $\mathcal{O}(\Delta t^4)$, global error is $\mathcal{O}(\Delta t^2)$.
- [ ] **(D)** Local error is $\mathcal{O}(\Delta t^4)$, global error is $\mathcal{O}(\Delta t^3)$.

In [8]:
ws.render_section('q04_04')

**Q4 · easy** · 💡

How many force evaluations per particle are required in each timestep of the Velocity Verlet algorithm?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Notice that the acceleration $\mathbf{a}(t)$ from the end of the previous step is reused at the beginning of the new step.

</div>
</details>

- [ ] **(A)** 1
- [ ] **(B)** 2
- [ ] **(C)** 3
- [ ] **(D)** 4

In [9]:
ws.render_section('q04_05')

**Q5 · medium** · 💡

When running the forward Euler integrator on a 1D harmonic oscillator in the Concept Cell, what happens to the total energy and phase space orbit?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Calculate $x_{n+1}^2 + v_{n+1}^2$ using $x_{n+1} = x_n + v_n \Delta t$ and $v_{n+1} = v_n - \omega^2 x_n \Delta t$.

</div>
</details>

- [ ] **(A)** Energy is exactly conserved and the orbit is a closed circle.
- [ ] **(B)** Energy decreases monotonically and the orbit spirals into the origin $(0, 0)$.
- [ ] **(C)** Energy grows monotonically with each step ($E_{n+1} = E_n(1 + \omega^2 \Delta t^2)$), causing the phase orbit to spiral outward indefinitely.
- [ ] **(D)** Energy oscillates periodically with zero net drift.

In [10]:
ws.render_section('q04_06')

**Q6 · medium** · 💡

Which of the following statements about the Leapfrog algorithm are TRUE? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about where velocities live relative to positions in Leapfrog, and how many force calls are needed.

</div>
</details>

- [ ] **(A)** Velocities are evaluated at half-integer timesteps $t + \frac{1}{2}\Delta t$, while positions are evaluated at integer timesteps $t$.
- [ ] **(B)** It requires four force evaluations per particle per timestep.
- [ ] **(C)** It produces mathematically identical position trajectories to the standard Verlet and Velocity Verlet algorithms.
- [ ] **(D)** Kinetic energy and potential energy cannot be evaluated at the exact same instant without an interpolation step.

In [11]:
ws.render_section('q04_07')

**Q7 · medium** · 💡

How does Runge-Kutta 4th order (RK4) compare to Velocity Verlet for long-time ($10^6$ steps) molecular dynamics simulations?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the distinction between local truncation error order and symplecticity (conservation of shadow Hamiltonian).

</div>
</details>

- [ ] **(A)** RK4 is superior for long simulations because its $\mathcal{O}(\Delta t^4)$ error guarantees zero energy drift forever.
- [ ] **(B)** RK4 is non-symplectic, causing secular energy drift (energy continually ramps up or down) over long times, and it requires 4 force evaluations per step versus 1 for Verlet.
- [ ] **(C)** RK4 is identical to Velocity Verlet in both computational cost and symplecticity.
- [ ] **(D)** RK4 cannot be applied to systems with velocity-independent forces.

In [12]:
ws.render_section('q04_08')

**Q8 · easy** · 💡

In Velocity Verlet, a particle starts at $x(0) = 1.0\text{ nm}$ with velocity $v(0) = 0.5\text{ nm/ps}$ and constant acceleration $a = -2.0\text{ nm/ps}^2$. Using timestep $\Delta t = 0.1\text{ ps}$, calculate its position $x(\Delta t)$ in nm.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Use $x(\Delta t) = x(0) + v(0)\Delta t + \frac{1}{2}a(0)\Delta t^2$.

</div>
</details>

_Numeric answer required (in nm)._

In [13]:
ws.render_section('q04_09')

**Q9 · medium** · 💡

Continuing from the previous problem, assuming the force remains constant so that $a(\Delta t) = -2.0\text{ nm/ps}^2$, calculate the final velocity $v(\Delta t)$ in nm/ps after the step.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Use $v(\Delta t) = v(0) + \frac{1}{2}(a(0) + a(\Delta t))\Delta t$.

</div>
</details>

_Numeric answer required (in nm/ps)._

In [14]:
ws.render_section('q04_10')

**Q10 · hard** · 💡

Let $\mathbf{M}$ be the Jacobian matrix of a single-step phase space transformation $(\mathbf{r}_{n+1}, \mathbf{p}_{n+1}) = \Phi_{\Delta t}(\mathbf{r}_n, \mathbf{p}_n)$. What is the necessary and sufficient condition for $\Phi_{\Delta t}$ to be a symplectic map?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Symplectic maps preserve the canonical Poisson bracket structure and phase space 2-form $\omega$.

</div>
</details>

- [ ] **(A)** $\mathbf{M}^T \mathbf{M} = \mathbf{I}$ (orthogonal matrix)
- [ ] **(B)** $\mathbf{M}^T \mathbf{J} \mathbf{M} = \mathbf{J}$, where $\mathbf{J} = \begin{pmatrix} 0 & \mathbf{I} \\ -\mathbf{I} & 0 \end{pmatrix}$ is the standard symplectic matrix
- [ ] **(C)** $\text{Tr}(\mathbf{M}) = 0$
- [ ] **(D)** All eigenvalues of $\mathbf{M}$ must be strictly real

In [15]:
ws.render_section('sec_shadow_hamiltonian_and_drift')

### Why Doesn't Verlet Drift in Energy?

In naive numerical integration of non-linear ODEs, one expects numerical errors to accumulate like a random walk or secular trend, causing the computed energy $E(t)$ to diverge systematically over long simulation times.

Yet in MD simulations lasting millions of steps, Velocity Verlet exhibits **bounded energy oscillations** with **zero secular drift**. This remarkable property is explained by **backward error analysis**:

A symplectic integrator does not follow the exact trajectory of the true Hamiltonian $\mathcal{H}(\mathbf{r}, \mathbf{p})$, but it follows the *exact* trajectory of a nearby **shadow Hamiltonian** (or modified Hamiltonian) $\widetilde{\mathcal{H}}$:

$$\widetilde{\mathcal{H}}(\mathbf{r}, \mathbf{p}) = \mathcal{H}(\mathbf{r}, \mathbf{p}) + \Delta t^2 \mathcal{H}_2(\mathbf{r}, \mathbf{p}) + \Delta t^4 \mathcal{H}_4(\mathbf{r}, \mathbf{p}) + \dots$$

For a 1D harmonic oscillator with $\mathcal{H} = \frac{1}{2}p^2 + \frac{1}{2}\omega^2 x^2$, the shadow Hamiltonian conserved exactly by Velocity Verlet is:

$$\widetilde{\mathcal{H}} = \frac{p^2}{2(1 - \frac{1}{4}\omega^2 \Delta t^2)} + \frac{1}{2}\omega^2 x^2$$

Because $\widetilde{\mathcal{H}}$ is a true constant of motion for the discrete trajectory, the true energy $\mathcal{H}$ cannot drift away! It merely oscillates around $\widetilde{\mathcal{H}}$ with amplitude $\mathcal{O}(\Delta t^2)$:

$$|\mathcal{H}(t) - \widetilde{\mathcal{H}}| \le C \Delta t^2$$

Non-symplectic integrators (such as standard Runge-Kutta or Euler) do not conserve any shadow Hamiltonian, which allows their energy to drift linearly with time $\sim t \Delta t^p$.


In [16]:
ws.render_section('concept_cutoff_discontinuity')

### Interactive Concept: Potential Truncation & Force Discontinuity

Examine the effect of truncating a Lennard-Jones potential at cutoff radius $r_c$.
Compare the unshifted potential $V(r)$, the shifted potential $V_{\text{shift}}(r) = V(r) - V(r_c)$, and the force $F(r) = -dV/dr$. Notice the delta-function force impulse that occurs when an unshifted potential is cut off abruptly.

**Key Insight:** Abruptly truncating $V(r)$ at $r_c$ creates a discontinuous potential step $\Delta V = V(r_c) \ne 0$. Taking the negative gradient produces an infinite impulsive force $-V(r_c)\delta(r - r_c)$ whenever an atom crosses $r_c$, destroying energy conservation. Shifting the potential $V_{\text{shift}}(r) = V(r) - V(r_c)$ eliminates the step in $V$, but leaves a jump in force $F(r_c^-) \ne 0$ unless force-shifting or switching functions are applied.

In [17]:
ws.render_section('q04_11')

**Q11 · medium** · 💡

Why do symplectic integrators like Velocity Verlet exhibit bounded energy fluctuations without long-term drift?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at backward error analysis and the modified Hamiltonian concept.

</div>
</details>

- [ ] **(A)** They use variable timesteps that automatically shrink whenever energy starts to drift.
- [ ] **(B)** They exactly conserve a nearby 'shadow' or modified Hamiltonian $\widetilde{\mathcal{H}} = \mathcal{H} + \mathcal{O}(\Delta t^2)$.
- [ ] **(C)** Rounding errors from IEEE 754 floating point cancel out the integration truncation errors.
- [ ] **(D)** They enforce an explicit Lagrange multiplier constraint on the total kinetic energy.

In [18]:
ws.render_section('q04_12')

**Q12 · hard** · 💡

For a harmonic oscillator with $\mathcal{H} = \frac{1}{2}p^2 + \frac{1}{2}\omega^2 x^2$, what happens to the shadow Hamiltonian denominator $(1 - \frac{1}{4}\omega^2 \Delta t^2)$ when $\omega \Delta t \to 2$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Examine the condition $\frac{1}{4}\omega^2 \Delta t^2 = 1 \implies \omega \Delta t = 2$.

</div>
</details>

- [ ] **(A)** It approaches 1, meaning the integrator becomes infinitely accurate.
- [ ] **(B)** It approaches 0, signaling a resonance divergence where the shadow Hamiltonian ceases to exist and the numerical orbit blows up.
- [ ] **(C)** It changes sign smoothly with no effect on trajectory stability.
- [ ] **(D)** It forces the position to remain zero.

In [19]:
ws.render_section('q04_13')

**Q13 · easy** · 💡

If you halve the timestep $\Delta t \to \Delta t / 2$ in a Velocity Verlet simulation, by what factor do the RMS energy fluctuations $\sigma_E$ decrease?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the scaling of the difference between true Hamiltonian and shadow Hamiltonian: $|\mathcal{H} - \widetilde{\mathcal{H}}| \sim \Delta t^2$.

</div>
</details>

- [ ] **(A)** Factor of 2
- [ ] **(B)** Factor of 4
- [ ] **(C)** Factor of 8
- [ ] **(D)** Factor of 16

In [20]:
ws.render_section('q04_14')

**Q14 · medium** · 💡

Why is the simple truncation potential $V_{\text{trunc}}(r) = V(r)$ for $r \le r_c$ and $0$ for $r > r_c$ unacceptable in microcanonical ($NVE$) MD simulations?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What is the derivative of a step function $\Theta(r_c - r)$?

</div>
</details>

- [ ] **(A)** It makes the forces zero everywhere.
- [ ] **(B)** The discontinuity in $V(r)$ at $r_c$ produces an infinite delta-function impulse force when particles cross $r_c$, causing severe energy non-conservation.
- [ ] **(C)** It doubles the computational complexity of the neighbor list.
- [ ] **(D)** It violates Newton's third law $\mathbf{F}_{ij} = -\mathbf{F}_{ji}$.

In [21]:
ws.render_section('q04_15')

**Q15 · easy** · 💡

How does the standard shifted potential $V_{\text{shift}}(r) = V(r) - V(r_c)$ resolve this energy discontinuity?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $V_{\text{shift}}(r)$ as $r \to r_c^-$.

</div>
</details>

- [ ] **(A)** It ensures that $V_{\text{shift}}(r_c) = 0$, making the potential continuous at the cutoff boundary.
- [ ] **(B)** It sets all forces to zero within the sphere of radius $r_c$.
- [ ] **(C)** It eliminates the need for periodic boundary conditions.
- [ ] **(D)** It alters the atomic forces at all distances $r < r_c$.

In [22]:
ws.render_section('q04_16')

**Q16 · medium** · 💡

While the shifted potential $V_{\text{shift}}(r) = V(r) - V(r_c)$ makes the potential energy continuous at $r_c$, what discontinuity remains?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Does shifting by a constant alter the derivative $V'(r_c)$?

</div>
</details>

- [ ] **(A)** The position of particles becomes discontinuous.
- [ ] **(B)** The force $F(r) = -dV/dr$ has a finite jump discontinuity at $r_c$ because $F(r_c^-) = -V'(r_c) \ne 0$ while $F(r_c^+) = 0$.
- [ ] **(C)** The total momentum is no longer conserved.
- [ ] **(D)** The mass of the particles becomes time-dependent.

In [23]:
ws.render_section('q04_17')

**Q17 · medium** · 💡

For a standard Lennard-Jones potential $V(r) = 4\epsilon \left[ (\sigma/r)^{12} - (\sigma/r)^6 \right]$ with $\epsilon = 1.0$ and $\sigma = 1.0$, calculate the unshifted potential value $V(r_c)$ at cutoff $r_c = 2.5\sigma$. (Enter as a signed decimal to 4 decimal places).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Calculate $(1/2.5)^{12} - (1/2.5)^6$ and multiply by 4.

</div>
</details>

_Numeric answer required._

In [24]:
ws.render_section('q04_18')

**Q18 · medium** · 💡

Calculate the magnitude of the force jump $|F(r_c)|$ in units of $\epsilon/\sigma$ at $r_c = 2.5\sigma$ for the Lennard-Jones potential with $\epsilon=1, \sigma=1$. (Recall $F(r) = 24\frac{\epsilon}{\sigma}\left[ 2(\sigma/r)^{13} - (\sigma/r)^7 \right]$).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute $r=2.5$ into $24[2(2.5)^{-13} - (2.5)^{-7}]$.

</div>
</details>

_Numeric answer required (in epsilon/sigma)._

In [25]:
ws.render_section('q04_19')

**Q19 · medium** · 💡

A force-shifted potential $V_{\text{SF}}(r)$ is constructed as $V_{\text{SF}}(r) = V(r) - V(r_c) - \left(\frac{dV}{dr}\right)_{r_c} (r - r_c)$. What two properties does $V_{\text{SF}}(r)$ satisfy at $r = r_c$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Differentiate $V_{\text{SF}}(r)$ with respect to $r$ and evaluate at $r=r_c$.

</div>
</details>

- [ ] **(A)** Both $V_{\text{SF}}(r_c) = 0$ and $F_{\text{SF}}(r_c) = -V_{\text{SF}}'(r_c) = 0$.
- [ ] **(B)** $V_{\text{SF}}(r_c) = 1$ and $F_{\text{SF}}(r_c) = -1$.
- [ ] **(C)** The potential becomes linear everywhere.
- [ ] **(D)** The force becomes independent of interatomic distance.

In [26]:
ws.render_section('q04_20')

**Q20 · hard** · 💡

Which of the following methods preserve phase-space volume (are symplectic)? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Remember: classical Verlet variants are symplectic; naive Euler and standard high-order Runge-Kutta methods are not.

</div>
</details>

- [ ] **(A)** Velocity Verlet
- [ ] **(B)** Leapfrog
- [ ] **(C)** Forward Euler
- [ ] **(D)** Standard 4th-Order Runge-Kutta (RK4)
- [ ] **(E)** Position Verlet

In [27]:
ws.render_section('sec_timestep_selection_and_resonance')

### How to Choose the Timestep $\Delta t$

In atomic simulations, atoms vibrate around equilibrium lattice positions or molecular covalent bonds. The fastest motion in the system dictates the required integration timestep $\Delta t$.

1. **Fastest Vibrational Period:** If the highest vibrational frequency in the system is $\omega_{\max}$ (e.g., C-H bond stretching with $\nu \approx 3000\text{ cm}^{-1} \approx 9\times 10^{13}\text{ s}^{-1}$, corresponding to period $T_{\text{vib}} \approx 11\text{ fs}$), the standard empirical rule of thumb is:

$$\Delta t \approx \frac{T_{\text{vib}}}{20}$$

For flexible C-H bonds, this mandates $\Delta t \approx 0.5\text{--}1.0\text{ fs}$. Constraining these fastest bonds with SHAKE/RATTLE allows $\Delta t$ to increase to $2.0\text{ fs}$ (governed by slower C-C stretching or bond angle bending).

2. **Verlet Resonance Stability Limit:** Consider the harmonic oscillator $\ddot{x} + \omega^2 x = 0$ integrated by Verlet. The recurrence relation is:

$$x_{n+1} - (2 - \omega^2 \Delta t^2) x_n + x_{n-1} = 0$$

Setting $x_n = \lambda^n$ yields the characteristic equation $\lambda^2 - 2(1 - \frac{1}{2}\omega^2 \Delta t^2)\lambda + 1 = 0$. For stable bounded oscillations, the roots must be complex conjugates of unit magnitude ($|\lambda| = 1$), which requires:

$$\left| 1 - \frac{1}{2}\omega^2 \Delta t^2 \right| < 1 \implies \omega \Delta t < 2 \iff \Delta t < \frac{2}{\omega} = \frac{T_{\text{vib}}}{\pi} \approx 0.318 T_{\text{vib}}$$

If $\Delta t > T_{\text{vib}}/\pi$, the roots $\lambda$ become real and one root satisfies $|\lambda| > 1$, leading to **exponential numerical blowup**!


In [28]:
ws.render_section('concept_timestep_stability_frontier')

### Interactive Concept: Timestep Stability Frontier & The Verlet Resonance Limit

Explore how relative energy drift rate depends on the chosen timestep $\Delta t / T_{\text{vib}}$.
Move the slider for `selected_dt` and observe the transition from safe quadratic scaling $\mathcal{O}(\Delta t^2)$, to elevated fluctuations, and finally to the catastrophic resonance divergence at $\Delta t_{\text{crit}} = T_{\text{vib}}/\pi \approx 0.318$.

**Key Insight:** Choosing $\Delta t \approx T_{\text{vib}}/20 = 0.05 T_{\text{vib}}$ places the simulation safely in the convergent regime where shadow Hamiltonian error is $< 10^{-4}$ and energy is conserved to within 0.01%. As $\Delta t$ approaches $T_{\text{vib}}/\pi \approx 0.318 T_{\text{vib}}$, resonance between the integrator step and physical vibration causes numerical instability.

In [29]:
ws.render_section('q04_21')

**Q21 · easy** · 💡

What is the theoretical upper stability limit $\Delta t_{\text{crit}}$ for integrating a harmonic mode with frequency $\omega$ using the Verlet algorithm?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Analyze the roots of $\lambda^2 - 2(1 - \frac{1}{2}\omega^2 \Delta t^2)\lambda + 1 = 0$.

</div>
</details>

- [ ] **(A)** $\Delta t_{\text{crit}} = 1/\omega$
- [ ] **(B)** $\Delta t_{\text{crit}} = 2/\omega = T_{\text{vib}}/\pi$
- [ ] **(C)** $\Delta t_{\text{crit}} = \pi/\omega$
- [ ] **(D)** $\Delta t_{\text{crit}} = 2\pi/\omega = T_{\text{vib}}$

In [30]:
ws.render_section('q04_22')

**Q22 · medium** · 💡

A water molecule has an O-H symmetric stretch frequency of $\nu = 1.1 \times 10^{14}\text{ s}^{-1}$ ($T_{\text{vib}} \approx 9.09\text{ fs}$). Using the rule of thumb $\Delta t = T_{\text{vib}}/20$, calculate the recommended maximum timestep in femtoseconds (fs).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Divide the period $9.09\text{ fs}$ by 20.

</div>
</details>

_Numeric answer required (in fs)._

In [31]:
ws.render_section('q04_23')

**Q23 · medium** · 💡

For the same O-H vibration with $T_{\text{vib}} = 9.09\text{ fs}$, what is the critical resonance blowup timestep $\Delta t_{\text{crit}} = T_{\text{vib}}/\pi$ in femtoseconds?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Divide $9.09\text{ fs}$ by $\pi \approx 3.14159$.

</div>
</details>

_Numeric answer required (in fs)._

In [32]:
ws.render_section('q04_24')

**Q24 · easy** · 💡

In classical liquid argon ($m = 39.95\text{ g/mol}$, $\epsilon/k_B = 120\text{ K}$, $\sigma = 0.34\text{ nm}$), typical vibrational periods of atoms in local cages are $T_{\text{vib}} \sim 1\text{ ps}$. What is the typical timestep used in Lennard-Jones liquid simulations?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Convert Lennard-Jones unit of time $\tau = \sqrt{m\sigma^2/\epsilon} \approx 2.15\text{ ps}$.

</div>
</details>

- [ ] **(A)** $0.001\text{ fs}$
- [ ] **(B)** $1\text{ to }5\text{ fs}$ (or $\approx 0.001\text{ to }0.005\,\tau_{\text{LJ}}$)
- [ ] **(C)** $100\text{ ps}$
- [ ] **(D)** $1\text{ ns}$

In [33]:
ws.render_section('q04_25')

**Q25 · medium** · 💡

According to the common simulation rule of thumb cited in Berendsen and course notes, what is an acceptable energy conservation criterion for a microcanonical ($NVE$) simulation?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the criteria for an integrator in slide 7 of 04.MD.pdf.

</div>
</details>

- [ ] **(A)** The total energy must remain strictly constant to within machine epsilon ($10^{-16}$).
- [ ] **(B)** Energy drift over the course of the simulation should remain less than 1% of the total kinetic energy ($|\Delta E| / K < 0.01$).
- [ ] **(C)** Potential energy should equal kinetic energy at every step.
- [ ] **(D)** Energy drift of 50% is acceptable as long as temperature is maintained by scaling.

In [34]:
ws.render_section('q04_26')

**Q26 · medium** · 💡

Which of the following consequences occur if the timestep $\Delta t$ is chosen TOO LARGE in an MD simulation? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about both physical particle overlaps and numerical integrator stability limits.

</div>
</details>

- [ ] **(A)** Atoms with high velocities may overlap severely in a single step, experiencing astronomical repulsive forces ($r^{-12}$ blowup).
- [ ] **(B)** High-frequency vibrational modes cross the $\omega \Delta t = 2$ stability threshold, causing exponential energy explosion.
- [ ] **(C)** The simulation completes more wall-clock time per physical nanosecond.
- [ ] **(D)** The phase space Jacobian determinant deviates significantly from 1 in non-symplectic integrators.

In [35]:
ws.render_section('q04_27')

**Q27 · hard** · 💡

In multi-timestep algorithms like RESPA (Reversible Reference System Propagator Algorithm), how are fast and slow forces treated?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about breaking the Liouville propagator into $e^{iL_{\text{fast}}\delta t}$ and $e^{iL_{\text{slow}}\Delta t}$.

</div>
</details>

- [ ] **(A)** All forces are evaluated at the same large timestep $\Delta t$.
- [ ] **(B)** Fast forces (covalent bonds/angles) are evaluated with a small timestep $\delta t$, while expensive slow forces (non-bonded LJ/Coulomb) are evaluated with a larger timestep $\Delta t = n\,\delta t$, using a symplectic operator factorization.
- [ ] **(C)** Fast forces are discarded completely.
- [ ] **(D)** Slow forces are evaluated using Monte Carlo while fast forces use MD.

In [36]:
ws.render_section('q04_28')

**Q28 · easy** · 💡

If a simulation of 100,000 steps is run with $\Delta t = 2.0\text{ fs}$, what is the total simulated physical trajectory duration in picoseconds (ps)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Multiply steps by $\Delta t$ and convert fs to ps ($1000\text{ fs} = 1\text{ ps}$).

</div>
</details>

_Numeric answer required (in ps)._

In [37]:
ws.render_section('q04_29')

**Q29 · medium** · 💡

In the concept cell 'Timestep Stability Frontier', what is the functional dependence of the energy drift rate on $\Delta t$ when $\Delta t$ is well below the resonance threshold?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the order of the leading term in the Velocity Verlet shadow Hamiltonian $\mathcal{H}_2 \Delta t^2$.

</div>
</details>

- [ ] **(A)** Linear: $|dE/dt| \propto \Delta t$
- [ ] **(B)** Quadratic: $|dE/dt| \propto \Delta t^2$
- [ ] **(C)** Exponential: $|dE/dt| \propto e^{\Delta t}$
- [ ] **(D)** Independent of $\Delta t$

In [38]:
ws.render_section('q04_30')

**Q30 · medium** · 💡

Why does freezing the bond lengths of hydrogen atoms using constraint algorithms (like SHAKE) allow researchers to double the MD timestep from $1\text{ fs}$ to $2\text{ fs}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What vibration is fastest in organic molecules containing hydrogen?

</div>
</details>

- [ ] **(A)** It eliminates the highest-frequency vibrational modes (~3000 cm^-1 C-H stretch), so the fastest remaining modes (angle bends ~1500 cm^-1) have roughly double the period.
- [ ] **(B)** It removes all kinetic energy from the simulation.
- [ ] **(C)** It changes the integrator from Verlet to Runge-Kutta.
- [ ] **(D)** It makes hydrogen atoms infinitely heavy.

In [39]:
ws.render_section('sec_reversibility_and_complexity')

### Exact Algebraic Time-Reversibility

Newton's equations of motion are invariant under time reversal $t \to -t$ and $\mathbf{v} \to -\mathbf{v}$. An integrator is **time-reversible** if reversing the velocities $\mathbf{v} \to -\mathbf{v}$ causes the trajectory to trace back its steps in exact reverse order.

In the Velocity Verlet algorithm:

$$\mathbf{r}(t+\Delta t) = \mathbf{r}(t) + \mathbf{v}(t)\Delta t + \frac{1}{2}\mathbf{a}(t)\Delta t^2$$
$$\mathbf{v}(t+\Delta t) = \mathbf{v}(t) + \frac{1}{2}[\mathbf{a}(t) + \mathbf{a}(t+\Delta t)]\Delta t$$

Replacing $\Delta t$ with $-\Delta t$ yields the exact algebraic inverse mapping. However, in practical floating-point simulations, two factors limit numerical reversibility:

1. **Finite-Precision Roundoff:** Non-associativity of IEEE 754 floating-point addition ($(a + b) + c \ne a + (b + c)$).
2. **Lyapunov Instability:** Many-body systems are chaotic. Two trajectories separated initially by $\delta r(0)$ diverge exponentially:

$$\delta r(t) \approx \delta r(0) e^{\lambda_{\text{max}} t}$$

where $\lambda_{\text{max}} > 0$ is the maximal Lyapunov exponent. After a Lyapunov time $\tau_L \sim 1/\lambda_{\text{max}}$, microscopic roundoff error is amplified to macroscopic separation, preventing long-time trajectory reversal even though macroscopic thermodynamics remains exact!

### Computational Complexity of Force Evaluation

A naive loop over all pairs in an $N$-atom system scales as $\mathcal{O}(N^2)$:

$$N_{\text{pairs}} = \frac{N(N - 1)}{2} \approx \frac{N^2}{2}$$

For short-range potentials truncated at $r_c$, two techniques reduce this complexity to $\mathcal{O}(N)$:

- **Verlet Neighbor Lists:** Each atom maintains a list of neighbors within a larger sphere $r_m = r_c + r_{\text{skin}}$. The list is rebuilt only when the maximum particle displacement exceeds $r_{\text{skin}}/2$, amortizing the search cost over many steps.
- **Cell Subdivision (Linked-Cell Lists):** The simulation box is divided into cells of width $d \ge r_c$. Each atom only checks neighbors in its own cell and the 26 adjacent neighboring cells, achieving strict $\mathcal{O}(N)$ scaling.


In [40]:
ws.render_section('concept_time_reversibility_lab')

### Interactive Concept: Time-Reversibility Lab & Velocity Inversion

Test the algebraic time-reversibility of the Velocity Verlet algorithm.
The simulation integrates $N_{\text{forward}}$ steps forward, inverts all particle velocities $\mathbf{v} \to -\mathbf{v}$, and integrates $N_{\text{forward}}$ steps backward. Adjust $N_{\text{forward}}$, $\Delta t$, and add a tiny perturbation $\epsilon$ to observe the Lyapunov divergence.

**Key Insight:** In the absence of perturbations, Velocity Verlet retraces its forward path back to the starting point to machine precision ($|x_f - x_0| \sim 10^{-14}$). However, introducing even a tiny perturbation $\epsilon \sim 10^{-5}$ causes chaotic trajectory divergence due to the positive Lyapunov exponent of the system.

In [41]:
ws.render_section('q04_31')

**Q31 · easy** · 💡

In an $N$-body system with pair potentials, how many pair distances $r_{ij}$ must be computed in a naive direct double sum without neighbor lists?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

How many distinct pairs can be chosen from $N$ items?

</div>
</details>

- [ ] **(A)** $N$
- [ ] **(B)** $N \log N$
- [ ] **(C)** $\frac{N(N - 1)}{2} \approx \frac{N^2}{2}$
- [ ] **(D)** $N^3$

In [42]:
ws.render_section('q04_32')

**Q32 · medium** · 💡

In a Verlet neighbor list with cutoff $r_c$ and skin buffer $r_{\text{skin}}$ (so list radius $r_m = r_c + r_{\text{skin}}$), what is the criterion that guarantees no pair can enter $r_c$ before the neighbor list is rebuilt?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

If particle $i$ moves by $\Delta r_i$ and particle $j$ moves by $\Delta r_j$, what is the maximum change in their separation distance $|\Delta r_{ij}|$?

</div>
</details>

- [ ] **(A)** The simulation has completed 100 timesteps.
- [ ] **(B)** The maximum displacement of any particle since the last update exceeds $r_{\text{skin}} / 2$ (or sum of maximum two displacements $> r_{\text{skin}}$).
- [ ] **(C)** The total kinetic energy changes by more than 5%.
- [ ] **(D)** At least one particle crosses a periodic boundary.

In [43]:
ws.render_section('q04_33')

**Q33 · medium** · 💡

A simulation of $N = 10,000$ atoms uses a naive $\mathcal{O}(N^2)$ pair loop. If the system size is increased to $N = 50,000$ atoms, by what factor does the pair calculation time increase?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Calculate $(50,000 / 10,000)^2$.

</div>
</details>

_Numeric answer required._

In [44]:
ws.render_section('q04_34')

**Q34 · medium** · 💡

In the cell linked-list method, the simulation volume is divided into cubic cells of width $d$. What is the minimum cell width $d$ required so that neighbors within $r_c$ need only be searched in the atom's own cell and the 26 immediate neighbor cells?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

If an atom is at the far boundary of a cell, how far can a particle be in the adjacent cell?

</div>
</details>

- [ ] **(A)** $d \ge r_c / 2$
- [ ] **(B)** $d \ge r_c$
- [ ] **(C)** $d \ge 2 r_c$
- [ ] **(D)** $d \ge \pi r_c$

In [45]:
ws.render_section('q04_35')

**Q35 · hard** · 💡

Why are MD trajectories in a classical many-body liquid unable to retrace their forward path when velocities are reversed after a long time ($t > 100\text{ ps}$), despite Velocity Verlet being algebraically time-reversible?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the discussion of chaos, Lyapunov instability, and roundoff error.

</div>
</details>

- [ ] **(A)** Verlet algorithm breaks time-reversal symmetry by having odd orders of $\Delta t$.
- [ ] **(B)** The positive Lyapunov exponent causes exponential growth $\sim e^{\lambda t}$ of tiny floating-point roundoff errors ($10^{-16}$), completely desynchronizing the microscopic reversed trajectory.
- [ ] **(C)** Newton's laws of motion are fundamentally not invariant under time reversal in three dimensions.
- [ ] **(D)** Potential cutoffs permanently delete particle information from memory.

In [46]:
ws.render_section('q04_36')

**Q36 · medium** · 💡

If chaotic divergence destroys exact microscopic time reversibility after a few picoseconds, why are equilibrium thermodynamic properties computed from MD simulations still valid?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review the quote from Berendsen on slide 9 and the concept of 'shadowing'.

</div>
</details>

- [ ] **(A)** The shadowing lemma guarantees that the computed numerical trajectory remains uniformly close to *some* true physical trajectory with slightly different initial conditions.
- [ ] **(B)** Thermodynamics only depends on the first timestep of a simulation.
- [ ] **(C)** Chaos only affects the potential energy, not the kinetic energy.
- [ ] **(D)** Floating point errors always average to zero identically.

In [47]:
ws.render_section('q04_37')

**Q37 · hard** · 💡

In a liquid simulation with maximal Lyapunov exponent $\lambda_{\text{max}} = 0.5\text{ ps}^{-1}$, an initial floating-point roundoff error is $\delta_0 = 10^{-15}\text{ nm}$. Approximately how long in picoseconds (ps) will it take for the trajectory separation $\delta(t) = \delta_0 e^{\lambda_{\text{max}} t}$ to grow to the size of an atomic radius $\delta(t) = 0.1\text{ nm}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Solve for $t$: $t = \frac{1}{\lambda} \ln(\delta / \delta_0)$.

</div>
</details>

_Numeric answer required (in ps)._

In [48]:
ws.render_section('q04_38')

**Q38 · medium** · 💡

In the concept cell 'Time-Reversibility Lab', when $\epsilon = 0$, what order of magnitude is the final position return error $|x_{\text{end}} - x_0|$ after 80 forward and 80 reverse steps?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Notice that for a small number of steps in double precision, roundoff error has not had enough Lyapunov time to grow to macroscopic scale.

</div>
</details>

- [ ] **(A)** Machine precision ($10^{-15}$ to $10^{-14}$)
- [ ] **(B)** Order 1 ($1.0$)
- [ ] **(C)** Order $10^5$
- [ ] **(D)** Zero exactly (integer arithmetic)

In [49]:
ws.render_section('q04_39')

**Q39 · medium** · 💡

Which of the following boundary conditions can be applied to classical molecular dynamics simulations? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review slide 11 of 04.MD.pdf on spatial scales and boundary condition possibilities.

</div>
</details>

- [ ] **(A)** Periodic boundary conditions (PBC) with minimum image convention to model bulk liquids and solids.
- [ ] **(B)** Free boundary conditions (isolated cluster or molecule in vacuum).
- [ ] **(C)** Stochastic thermal walls or fixed repulsive walls to model confinement in a nanopore.
- [ ] **(D)** Klein bottle boundary conditions where particles flip chirality upon crossing.

In [50]:
ws.render_section('q04_40')

**Q40 · easy** · 💡

Why do MD simulations of bulk fluids and solids almost universally employ Periodic Boundary Conditions (PBC)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What fraction of atoms sit on the surface of a cube of $1000$ atoms ($10 \times 10 \times 10$)?

</div>
</details>

- [ ] **(A)** To eliminate surface and boundary effects so a small system of $10^3\text{--}10^6$ particles mimics an infinite bulk macroscopic system.
- [ ] **(B)** To force all particles to remain completely stationary.
- [ ] **(C)** To avoid computing Newton's equations of motion.
- [ ] **(D)** Because computers cannot store negative coordinates.

In [51]:
ws.render_section('q04_41')

**Q41 · hard** · 💡

Gear predictor-corrector integrators were historically popular in MD because they offer $\mathcal{O}(\Delta t^5)$ local truncation errors. Why have they been almost entirely superseded by Velocity Verlet in modern simulations?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review slide 8: 'Higher Order Methods?' in 04.MD.pdf.

</div>
</details>

- [ ] **(A)** Gear integrators require too many square roots.
- [ ] **(B)** Gear integrators are non-symplectic and non-reversible, leading to slow secular energy drift and poor phase-space volume conservation over long trajectories.
- [ ] **(C)** Gear integrators cannot handle more than 10 particles.
- [ ] **(D)** Gear integrators only work for 1D systems.

In [52]:
ws.render_section('q04_42')

**Q42 · medium** · 💡

A cubic simulation box of edge length $L = 5.0\text{ nm}$ contains particles interacting via a potential with cutoff $r_c = 1.2\text{ nm}$. If the box is subdivided into linked cells of equal size $d \ge r_c$, what is the maximum integer number of cell divisions $M$ along each dimension (such that $L/M \ge r_c$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

We need $d = L/M \ge r_c \implies M \le L / r_c$.

</div>
</details>

_Numeric answer required._

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()